In [ ]:
# Configure environment for VS Code on Windows + Python 3.15
import asyncio, sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import matplotlib
matplotlib.use('Agg')
print('Environment configured.')


# 03 — Preprocessing Mathematics

Learn how descriptive statistics and scaling relate to ML inputs using the uploaded dataset.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data_loading import load_uploaded
from src.preprocessing import descriptive_statistics, split_aware_standardise
df = load_uploaded()
display(descriptive_statistics(df))

## Mean, median, variance, standard deviation and IQR

Mean is the arithmetic average; median is the midpoint; variance measures squared spread; standard deviation is the square root of variance; IQR = Q3 − Q1 and is less sensitive to extremes.

In [ ]:
numeric = df.select_dtypes(include=np.number)
display(numeric.agg(["mean", "median", "var", "std", "min", "max"]).T)

## Standardisation and data leakage

For feature value x, training mean μ and training standard deviation σ: z = (x − μ) / σ. Split before fitting a scaler; fitting it on all rows leaks information about the held-out test set. Scaling is often useful for Logistic Regression, while tree models usually do not require it.

In [ ]:
target = "Outcome"
X = df.select_dtypes(include=np.number).drop(columns=[target], errors="ignore")
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026, stratify=y)
X_train_scaled, X_test_scaled, scaler = split_aware_standardise(X_train, X_test)
summary = pd.DataFrame({"mean_before": X_train.mean(), "std_before": X_train.std(ddof=0), "mean_after": X_train_scaled.mean(), "std_after": X_train_scaled.std(ddof=0)})
display(summary.round(4))
print("Train/test rows:", len(X_train), len(X_test))

## Correlation caveats

Pearson correlation measures linear association from −1 to +1. It can miss nonlinear relationships, be affected by outliers and be misleading for arbitrary category codes. Correlation does not establish causation. Use pipelines to fit imputers, scalers and feature selectors inside training folds.